# 부록. Pydantic / TypedDict 치트시트

타입 코드를 보다가 `Literal`, `Field`, `NotRequired` 같은 이름이 헷갈릴 때 보는 참고표입니다.  
외워야 하는 자료가 아니라, 필요한 표현을 찾는 용도로 사용하세요.


## TypedDict 계열

`TypedDict`는 dict 형태의 데이터가 어떤 키를 갖는지 표시하는 타입 힌트입니다. LangGraph state처럼 여러 단계에서 값이 추가되는 dict를 설명할 때 자주 씁니다.

기본 `TypedDict`에서 `a: str`은 `a` 필드가 반드시 있어야 하고, 그 값이 문자열이라는 뜻입니다. `a: Required[str]`도 같은 의미를 명시적으로 적은 표현입니다.

```python
class State(TypedDict):
    a: str  # 필수 필드, 값은 문자열
    b: Required[str]  # 필수 필드임을 명시, 값은 문자열
    answer: NotRequired[str]  # 선택 필드, 값은 문자열
```

아래 표는 `TypedDict` 안에서 자주 함께 쓰는 표현입니다.

| 이름 | 쉽게 말하면 | 예시 |
|---|---|---|
| `NotRequired[str]` | 이 필드는 없어도 됩니다. 값이 있으면 문자열이어야 합니다. | `answer: NotRequired[str]` |
| `Required[str]` | 이 필드는 반드시 있어야 합니다. 값은 문자열이어야 합니다. | `id: Required[str]` |
| `total=False` | 이 `TypedDict`에 적은 필드들을 기본적으로 선택 필드로 봅니다. | `class State(TypedDict, total=False): ...` |
| `Literal[...]` | 정해진 값 중 하나만 쓸 수 있다는 뜻입니다. | `route: Literal["auto", "human"]` |
| `list[str]` | 문자열이 여러 개 들어가는 리스트입니다. | `messages: list[str]` |
| `dict[str, int]` | key는 문자열, value는 정수인 dict입니다. | `scores: dict[str, int]` |
| <code>str &#124; None</code> | 문자열이거나 `None`일 수 있다는 뜻입니다. | <code>order_id: str &#124; None</code> |


In [ ]:
from typing import Literal
from typing_extensions import NotRequired, TypedDict

class SupportState(TypedDict):  # dict 형태의 state 구조를 정의합니다.
    request: str  # 처음부터 필요한 입력값입니다.
    category: NotRequired[Literal["policy", "shipping", "general"]]  # 나중에 추가될 수 있는 분류값입니다.
    needs_review: NotRequired[bool]  # 검토 필요 여부도 나중에 채울 수 있습니다.
    final_answer: NotRequired[str]  # 최종 답변은 마지막 단계에서 생깁니다.

state: SupportState = {"request": "쿠폰 환불 요청입니다."}
state["category"] = "policy"  # 노드 실행 후 state에 값을 추가하는 예시입니다.
state["needs_review"] = True
state


## Pydantic 기본 도구

`TypedDict`는 dict의 모양을 코드에 표시하는 타입 힌트입니다. 실행 중에 값이 맞는지 직접 검사하지는 않습니다.

Pydantic의 `BaseModel`은 한 단계 더 나아가, 실제로 들어온 값이 정해둔 형식에 맞는지 검사합니다. 값이 빠졌거나 타입이 맞지 않으면 오류로 알려주고, 통과한 값은 모델 객체로 다룰 수 있습니다.

그래서 LangGraph state처럼 중간에 계속 바뀌는 dict에는 `TypedDict`가 가볍고, LLM 출력이나 Tool 입력처럼 형식을 확실히 맞춰야 하는 데이터에는 Pydantic이 더 적합합니다.

| 이름 | 쉽게 말하면 | 예시 |
|---|---|---|
| `BaseModel` | 검사할 데이터 모양을 class로 만듭니다. | `class Inquiry(BaseModel): ...` |
| `Field` | 필드 설명, 기본값, 제한 조건을 붙입니다. | `Field(description="문의 주제")` |
| `model_validate` | dict를 Pydantic 모델로 바꿔 봅니다. | `Inquiry.model_validate(data)` |
| `model_dump` | Pydantic 모델을 다시 dict로 바꿉니다. | `item.model_dump()` |
| `model_dump_json` | Pydantic 모델을 JSON 문자열로 바꿉니다. | `item.model_dump_json()` |
| `model_json_schema` | 이 모델이 어떤 모양인지 schema로 봅니다. | `Inquiry.model_json_schema()` |


## Field 옵션

`Field`는 필드에 설명이나 조건을 붙일 때 씁니다.

| 옵션 | 쉽게 말하면 | 예시 |
|---|---|---|
| `description` | 이 필드가 무엇인지 설명합니다. | `Field(description="문의 요약")` |
| `default` | 값이 없을 때 쓸 기본값입니다. | `Field(default="general")` |
| `default_factory` | 새 기본값을 매번 만들어줍니다. 리스트에 자주 씁니다. | `Field(default_factory=list)` |
| `ge` | 이 값 이상이어야 합니다. | `Field(ge=0)` |
| `gt` | 이 값보다 커야 합니다. | `Field(gt=0)` |
| `le` | 이 값 이하여야 합니다. | `Field(le=5)` |
| `lt` | 이 값보다 작아야 합니다. | `Field(lt=10)` |
| `min_length` | 문자열이 이 길이보다 짧으면 안 됩니다. | `Field(min_length=1)` |
| `max_length` | 문자열이 이 길이보다 길면 안 됩니다. | `Field(max_length=200)` |
| `pattern` | 문자열이 정해둔 형식과 맞아야 합니다. | `Field(pattern=r"^ORD-\\d{4}$")` |
| `alias` | 바깥에서 들어오는 이름이 다를 때 씁니다. | `Field(alias="orderId")` |


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class CustomerInquiry(BaseModel):  # 검증할 데이터 구조를 정의합니다.
    topic: Literal["배송", "교환", "환불", "쿠폰"] = Field(description="문의 주제")
    urgency: Literal["low", "medium", "high"] = Field(default="medium", description="긴급도")  # 값이 없으면 medium을 사용합니다.
    order_id: str | None = Field(default=None, description="주문번호가 있으면 추출")
    summary: str = Field(description="문의 요약", min_length=1, max_length=200)
    tags: list[str] = Field(default_factory=list, description="검색이나 분류에 쓸 태그")  # 빈 리스트 기본값은 default_factory를 씁니다.

inquiry = CustomerInquiry.model_validate({  # dict를 Pydantic 모델로 검증합니다.
    "topic": "배송",
    "order_id": "ORD-1001",
    "summary": "배송이 지연되어 오늘 안에 답변을 요청했습니다.",
    "tags": ["배송지연", "긴급"],
})

print(inquiry)
print(inquiry.model_dump())  # 모델을 다시 dict로 바꿉니다.


## TypedDict vs BaseModel

| 비교 | `TypedDict` | `BaseModel` |
|---|---|---|
| 한 줄 요약 | dict 모양을 적어둡니다. | 값이 맞는지 실제로 검사합니다. |
| 실행 중 검사 | 하지 않습니다. | 합니다. |
| 기본값 | 직접 넣어야 합니다. | `Field(default=...)`로 넣을 수 있습니다. |
| 잘 맞는 곳 | LangGraph state | structured output, Tool 입력 |
| 장점 | 가볍고 단순합니다. | 값이 틀리면 바로 알 수 있습니다. |

흐름 중간에 계속 바뀌는 state는 `TypedDict`, LLM 출력처럼 형식을 맞춰 받아야 하는 값은 `BaseModel`이 편합니다.


## LangChain structured output 예시

Pydantic 모델을 `response_format`에 넣으면, 모델 답변을 정해진 구조로 받을 수 있습니다. 결과는 `structured_response`에서 확인합니다.


In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class InquirySummary(BaseModel):  # LLM이 맞춰야 할 출력 구조입니다.
    topic: Literal["배송", "교환", "환불", "쿠폰", "기타"] = Field(description="문의 주제")
    order_id: str | None = Field(default=None, description="주문번호")
    summary: str = Field(description="한 문장 요약")

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
agent = create_agent(
    model=model,
    tools=[],
    response_format=InquirySummary,  # 응답을 InquirySummary 구조로 받습니다.
    system_prompt="고객 문의를 정해진 스키마로 추출하세요.",
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "ORD-1001 배송이 늦어서 오늘 안에 확인받고 싶어요."}
    ]
})

print(result["structured_response"])  # 구조화된 응답은 이 키에서 확인합니다.

## 자주 쓰는 조합

| 하고 싶은 것 | 쓰는 표현 | 뜻 |
|---|---|---|
| 정해진 분류값만 받기 | `topic: Literal["배송", "교환", "환불", "쿠폰"]` | 네 값 중 하나만 허용합니다. |
| 주문번호가 없을 수도 있게 하기 | <code>order_id: str &#124; None = None</code> | 문자열이거나 `None`입니다. |
| 필드 설명 붙이기 | `summary: str = Field(description="문의 요약")` | LLM이 필드 의미를 더 잘 이해합니다. |
| 0 이상 숫자만 받기 | `amount: int = Field(ge=0)` | 음수는 허용하지 않습니다. |
| 빈 리스트로 시작하기 | `tags: list[str] = Field(default_factory=list)` | 새 리스트를 기본값으로 만듭니다. |
| 나중에 생기는 state 값 | `answer: NotRequired[str]` | 처음 state에는 없어도 됩니다. |
| 분기 이름 제한하기 | `route: Literal["auto", "human"]` | 오타를 줄입니다. |
| 바깥 이름과 Python 이름 맞추기 | `order_id: str = Field(alias="orderId")` | 입력은 `orderId`, 코드에서는 `order_id`로 씁니다. |


## 빠른 체크

| 헷갈릴 때 | 보면 되는 것 |
|---|---|
| dict state 모양을 적고 싶다 | `TypedDict` |
| 나중에 생기는 state 키가 있다 | `NotRequired` |
| 값 후보를 몇 개로 제한하고 싶다 | `Literal` |
| LLM 출력 형식을 정하고 싶다 | `BaseModel` |
| 필드 설명을 붙이고 싶다 | `Field(description=...)` |
| 기본값을 넣고 싶다 | `Field(default=...)` |
| 빈 리스트 기본값이 필요하다 | `Field(default_factory=list)` |
| 값 범위를 제한하고 싶다 | `Field(ge=0)`, `Field(le=5)` |
